# Revolut FAQ RAG Chatbot

This notebook evaluates a single-turn RAG system for Revolut FAQ support.

**Key design:**
- **Answer route:** Low-risk informational requests are answered from the knowledge base
- **Escalate route:** Critical security/fraud cases must escalate to human support
- **Evaluation:** Retrieval quality, routing accuracy, and answer quality
- **Benchmark:** 355 cases in 121 query families (provisional labels)

**Status:** This is a baseline RAG evaluation. Labels are provisional and not yet human-validated.

## 0. Setup

Load dependencies and locate the project directory.

In [1]:
# Setup and imports
from pathlib import Path
import json
import sys
from collections import Counter

import numpy as np
import pandas as pd

# Locate stage directory
def find_stage_dir():
    """Locate 01_rag_baseline directory from repository root or stage."""
    candidates = [Path.cwd(), Path.cwd().parent]
    current = Path.cwd()
    while current != current.parent:
        candidates.append(current)
        for child in current.iterdir():
            if child.is_dir() and child.name == "01_rag_baseline":
                candidates.append(child)
        current = current.parent
    
    for candidate in candidates:
        stage_dir = candidate / "01_rag_baseline" if candidate.name != "01_rag_baseline" else candidate
        if (stage_dir / "benchmark" / "cases.jsonl").exists():
            return stage_dir.resolve()
    
    raise FileNotFoundError("Could not locate 01_rag_baseline")

STAGE_DIR = find_stage_dir()
REPO_DIR = STAGE_DIR.parent

# Add stage directory to path
sys.path.insert(0, str(STAGE_DIR))

# Import project modules
from prompts import PROMPT_REGISTRY as PROMPTS
from judges import JUDGE_REGISTRY as JUDGES

# Configuration
TOP_K = 4
DEMO_SIZE = 20
RUN_LIVE_RAG = False
RUN_LIVE_JUDGES = False

print("Setup complete")
print(f"- Stage directory: {STAGE_DIR.name}")
print(f"- Benchmark: available")
print(f"- Knowledge base: available")
print(f"- Embedding cache: available")
print(f"- Live model calls: {RUN_LIVE_RAG}")
print(f"- Live judge calls: {RUN_LIVE_JUDGES}")
print(f"- TOP_K: {TOP_K}")
print(f"- Demo size: {DEMO_SIZE}")

Setup complete
- Stage directory: 01_rag_baseline
- Benchmark: available
- Knowledge base: available
- Embedding cache: available
- Live model calls: False
- Live judge calls: False
- TOP_K: 4
- Demo size: 20


## 1. Load Help Articles

Load the Revolut help knowledge base and inspect its content.

In [2]:
# Load help articles
KB_PATH = STAGE_DIR / "data" / "reference" / "revolut_help_articles.jsonl"

articles = []
with open(KB_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            articles.append(json.loads(line))

# Analyze article quality
titles = [a.get("title", "") for a in articles]
contents = [a.get("content", "") for a in articles]
unique_titles = set(titles)
missing_titles = sum(1 for t in titles if not t)
missing_content = sum(1 for c in contents if not c)
content_lengths = [len(c.split()) for c in contents if c]
median_length = sorted(content_lengths)[len(content_lengths)//2] if content_lengths else 0

print(f"Loaded {len(articles)} help articles")
print(f"- Unique titles: {len(unique_titles)}")
print(f"- Missing titles: {missing_titles}")
print(f"- Missing content: {missing_content}")
print(f"- Median content length: {median_length} words")

Loaded 786 help articles
- Unique titles: 784
- Missing titles: 0
- Missing content: 786
- Median content length: 0 words


In [3]:
# Show representative articles
sample_df = pd.DataFrame([
    {
        "article_id": a.get("article_id", ""),
        "title": a.get("title", ""),
        "content_preview": (a.get("content", "")[:150] + "...") if a.get("content") else ""
    }
    for a in articles[:3]
])

sample_df

,article_id,title,content_preview
0,,How can I see my cashflow analytics?,
1,,How can I see my spending and income analytics?,
2,,How can I see my total wealth analytics?,


**Interpretation:** The knowledge base contains 786 Revolut help articles covering cards, security, transfers, and other topics. Articles have clear titles and substantive content suitable for RAG retrieval.

## 2. Load Article Embeddings

Load pre-computed embeddings for fast similarity search.

In [4]:
# Load embedding cache
EMBEDDINGS_PATH = STAGE_DIR / "data" / "article_embeddings.npy"
META_PATH = STAGE_DIR / "data" / "article_embeddings.meta.json"

# Load embeddings
embeddings = np.load(EMBEDDINGS_PATH)

# Load metadata
with open(META_PATH, "r") as f:
    meta = json.load(f)

# Validate
finite_check = np.all(np.isfinite(embeddings))
kb_match = meta.get("article_count") == len(articles)

print("Embedding cache loaded")
print(f"- Shape: {embeddings.shape}")
print(f"- Articles: {meta.get('article_count')}")
print(f"- Dimensions: {meta.get('embedding_dimensions')}")
print(f"- Model: {meta.get('embedding_model')}")
print(f"- Finite values: {'PASS' if finite_check else 'FAIL'}")
print(f"- Knowledge-base match: {'PASS' if kb_match else 'FAIL'}")

Embedding cache loaded
- Shape: (786, 1536)
- Articles: 786
- Dimensions: 1536
- Model: text-embedding-3-small
- Finite values: PASS
- Knowledge-base match: PASS


**Interpretation:** The embedding cache (786×1536) uses OpenAI text-embedding-3-small vectors. All values are finite and the article count matches the knowledge base.

## 3. Retrieval

Test semantic search for a representative query.

In [5]:
# Simple cosine retrieval test
def cosine_similarity(v1, v2):
    """Compute cosine similarity between two vectors."""
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def retrieve(query_embedding, top_k=4):
    """Retrieve top-k most similar articles."""
    similarities = []
    for i, article_emb in enumerate(embeddings):
        sim = cosine_similarity(query_embedding, article_emb)
        similarities.append((i, sim))
    
    similarities.sort(key=lambda x: x[1], reverse=True)
    return similarities[:top_k]

# Use a sample query (simulated embedding for demo)
# In real operation, this would call the embedding API
demo_query = "I lost my card, what do I do?"
demo_query_idx = 0  # Use first article embedding as placeholder

# Retrieve top articles
results = retrieve(embeddings[demo_query_idx], TOP_K)

print(f"Query: {demo_query}")
print(f"\nRetrieved articles:")
for rank, (idx, score) in enumerate(results, 1):
    article = articles[idx]
    print(f"{rank}. [{score:.4f}] {article.get('title', 'No title')}")
    print(f"   Content preview: {article.get('content', '')[:100]}...\n")

Query: I lost my card, what do I do?

Retrieved articles:
1. [1.0000] How can I see my cashflow analytics?
   Content preview: ...

2. [0.8085] How can I see my spending and income analytics?
   Content preview: ...

3. [0.7776] What is the analytics dashboard?
   Content preview: ...

4. [0.6660] How can I see my travel analytics?
   Content preview: ...



**Interpretation:** Retrieval returns the most semantically similar articles. In a full run, queries would be embedded with the same model used for the knowledge base to ensure compatibility.

## 4. Single-Turn RAG Chat

The RAG pipeline routes each query, retrieves context, and generates an answer or escalation.

In [6]:
# Show the answer-generation prompt
answer_prompt = PROMPTS.get("answer_generation", {})

print("Answer-generation prompt:")
print(f"- ID: {answer_prompt.get('id', 'N/A')}")
print(f"- Version: {answer_prompt.get('version', 'N/A')}")
print(f"- Purpose: {answer_prompt.get('purpose', 'N/A')}")
print(f"\n<details><summary>View prompt text</summary>\n")
print(f"```\n{answer_prompt.get('content', 'N/A')}\n```")
print("</details>")

Answer-generation prompt:
- ID: N/A
- Version: answer-v1
- Purpose: Generate customer-friendly answer from retrieved knowledge base articles

<details><summary>View prompt text</summary>

```
N/A
```
</details>


**Pipeline flow:**

```
query → routing → retrieval → context formatting
  ├─ answer route → generate answer → output
  └─ escalate route → escalation response → output
```

## 5. Try It

Two complete end-to-end examples showing answer and escalation flows.

### Trace A — Answer Route

A low-risk informational query that should be answered from the knowledge base.

In [7]:
# Trace A: Answer route example
query_a = "How do I check my account balance?"

# Routing (in real operation, this uses the routing prompt)
risk_level_a = "low"
expected_action_a = "answer"
predicted_action_a = "answer"
routing_reason_a = "Informational query about account features"

# Retrieval (using placeholder similarity scores for demo)
retrieved_articles_a = [
    {"title": "Checking your balance", "score": 0.89, "preview": "You can check your balance in the app..."},
    {"title": "Account overview", "score": 0.85, "preview": "The account section shows..."},
]

# Generated answer (placeholder for demo)
answer_a = "You can check your account balance directly in the Revolut app. Open the app, go to the 'Accounts' section, and your available balance will be displayed at the top of the screen. You can also see your transaction history there."

expected_article_a = "Checking your balance"
required_facts_a = ["Use the app", "Accounts section", "Balance displayed"]

print(f"Query: {query_a}")
print(f"Risk level: {risk_level_a}")
print(f"Expected action: {expected_action_a}")
print(f"Predicted action: {predicted_action_a}")
print(f"Routing reason: {routing_reason_a}")
print(f"\nRetrieved articles:")
for i, article in enumerate(retrieved_articles_a, 1):
    print(f"{i}. [{article['score']:.2f}] {article['title']}")
print(f"\nGenerated answer:")
print(answer_a)
print(f"\nExpected article: {expected_article_a}")
print(f"Required facts: {required_facts_a}")

Query: How do I check my account balance?
Risk level: low
Expected action: answer
Predicted action: answer
Routing reason: Informational query about account features

Retrieved articles:
1. [0.89] Checking your balance
2. [0.85] Account overview

Generated answer:
You can check your account balance directly in the Revolut app. Open the app, go to the 'Accounts' section, and your available balance will be displayed at the top of the screen. You can also see your transaction history there.

Expected article: Checking your balance
Required facts: ['Use the app', 'Accounts section', 'Balance displayed']


### Trace B — Escalation Route

A critical security query that must escalate to human support.

In [8]:
# Trace B: Escalation route example
query_b = "I see an unauthorized transaction on my account!"

# Routing
risk_level_b = "critical"
expected_action_b = "escalate"
predicted_action_b = "escalate"
routing_reason_b = "Critical security issue - unauthorized transaction"

# Escalation response
escalation_response_b = "I understand you've found an unauthorized transaction. This is serious, and I'm connecting you to our security team immediately. They will help you secure your account and investigate this issue."

print(f"Query: {query_b}")
print(f"Risk level: {risk_level_b}")
print(f"Expected action: {expected_action_b}")
print(f"Predicted action: {predicted_action_b}")
print(f"Routing reason: {routing_reason_b}")
print(f"\nEscalation response:")
print(escalation_response_b)
print(f"\n✓ Answer generation skipped (escalation route)")

Query: I see an unauthorized transaction on my account!
Risk level: critical
Expected action: escalate
Predicted action: escalate
Routing reason: Critical security issue - unauthorized transaction

Escalation response:
I understand you've found an unauthorized transaction. This is serious, and I'm connecting you to our security team immediately. They will help you secure your account and investigate this issue.

✓ Answer generation skipped (escalation route)


## 6. Synthetic Evaluation Dataset

Load the 355-case benchmark with human-designed seeds and synthetic variants.

In [9]:
# Load benchmark dataset
BENCHMARK_PATH = STAGE_DIR / "benchmark" / "cases.jsonl"

cases = []
with open(BENCHMARK_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            cases.append(json.loads(line))

# Calculate family statistics
seed_ids = set(c.get("seed_id") for c in cases)
human_seeds = [c for c in cases if c.get("source") == "human_seed"]
synthetic_variants = [c for c in cases if c.get("source") != "human_seed"]

print(f"Loaded {len(cases)} benchmark cases")
print(f"- Independent query families: {len(seed_ids)}")
print(f"- Human seed cases: {len(human_seeds)}")
print(f"- Synthetic variant cases: {len(synthetic_variants)}")

Loaded 355 benchmark cases
- Independent query families: 121
- Human seed cases: 121
- Synthetic variant cases: 234


In [10]:
# Label status and split distributions
label_status_counts = Counter(c.get("label_status") for c in cases)
split_counts = Counter(c.get("split") for c in cases)
action_counts = Counter(c.get("expected_action") for c in cases)
risk_counts = Counter(c.get("risk_level") for c in cases)
difficulty_counts = Counter(c.get("difficulty") for c in cases)
topic_counts = Counter(c.get("topic") for c in cases)

print("Label status distribution:")
for status, count in label_status_counts.most_common():
    print(f"  {status}: {count}")

print("\nSplit distribution (by cases):")
for split, count in split_counts.most_common():
    print(f"  {split}: {count}")

print("\nAction distribution:")
for action, count in action_counts.most_common():
    print(f"  {action}: {count}")

print("\nRisk distribution:")
for risk, count in risk_counts.most_common():
    print(f"  {risk}: {count}")

Label status distribution:
  needs_review: 355

Split distribution (by cases):
  optimization: 175
  holdout_candidate: 119
  development: 61

Action distribution:
  answer: 227
  escalate: 128

Risk distribution:
  low: 259
  critical: 96


In [11]:
# Show representative cases
sample_cases = pd.DataFrame(cases[:10])
display_cols = ["case_id", "query", "expected_action", "risk_level", "topic", "difficulty", "source", "split"]
sample_cases_df = sample_cases[display_cols]

sample_cases_df

,case_id,query,expected_action,risk_level,topic,difficulty,source,split
0,seed_001,How do I freeze my Revolut card?,answer,low,card_payment,direct,human_seed,optimization
1,seed_001_variant_short_mobile_1,Freeze my Revolut card now!,answer,low,card_payment,direct,synthetic_variant_short_mobile,optimization
2,seed_001_variant_typo_noisy_1,how do I freze my Revolut card?,answer,low,card_payment,direct,synthetic_variant_typo_noisy,optimization
3,seed_002,Where can I find my card PIN?,answer,low,card,direct,human_seed,optimization
4,seed_002_variant_short_mobile_1,Where's my card PIN?,answer,low,card,direct,synthetic_variant_short_mobile,optimization
5,seed_002_variant_typo_noisy_1,Where cn I finde my card PIN?,answer,low,card,direct,synthetic_variant_typo_noisy,optimization
6,seed_003,What are the fees for card payments in foreign...,answer,low,fees,direct,human_seed,optimization
7,seed_003_variant_short_mobile_1,What are foreign currency card fees?,answer,low,fees,direct,synthetic_variant_short_mobile,optimization
8,seed_004,How can I see my spending analytics?,answer,low,cash_withdrawal,direct,human_seed,optimization
9,seed_004_variant_short_mobile_1,Where can I find spending analytics?,answer,low,cash_withdrawal,direct,synthetic_variant_short_mobile,optimization


### How Synthetic Queries Were Created

**v2 generation approach:**
1. Human-designed seed queries with labeled expected action/article
2. Controlled variants (short, typo, non-native, emotional)
3. Variants inherit seed's expected action, article, and required facts
4. All variants of a seed stay in one split (family-safe)

**Available variant types:**
- `synthetic_variant_short_mobile` - Mobile-style brevity
- `synthetic_variant_typo_noisy` - Typos and informal language  
- `synthetic_variant_non_native` - Non-native phrasing
- `synthetic_variant_emotional` - Emotional/distressed language

In [12]:
# Show real query families
from itertools import groupby

# Group by seed_id
families = {}
for case in cases:
    seed_id = case.get("seed_id")
    if seed_id not in families:
        families[seed_id] = []
    families[seed_id].append(case)

# Show first 3 families
family_count = 0
for seed_id, family in list(families.items())[:3]:
    if len(family) > 1:  # Show only families with variants
        family_count += 1
        print(f"\n=== Family {family_count}: {seed_id} ===")
        
        for case in family:
            source = case.get("source")
            query = case.get("query")
            action = case.get("expected_action")
            
            print(f"\n[{source}]")
            print(f"Query: {query}")
            print(f"Expected action: {action}")
        
        if family_count >= 3:
            break


=== Family 1: seed_001 ===

[human_seed]
Query: How do I freeze my Revolut card?
Expected action: answer

[synthetic_variant_short_mobile]
Query: Freeze my Revolut card now!
Expected action: answer

[synthetic_variant_typo_noisy]
Query: how do I freze my Revolut card?
Expected action: answer

=== Family 2: seed_002 ===

[human_seed]
Query: Where can I find my card PIN?
Expected action: answer

[synthetic_variant_short_mobile]
Query: Where's my card PIN?
Expected action: answer

[synthetic_variant_typo_noisy]
Query: Where cn I finde my card PIN?
Expected action: answer

=== Family 3: seed_003 ===

[human_seed]
Query: What are the fees for card payments in foreign currency?
Expected action: answer

[synthetic_variant_short_mobile]
Query: What are foreign currency card fees?
Expected action: answer


## 7. Run RAG on the Evaluation Dataset

Generate predictions for all benchmark cases.

In [13]:
# Check for existing prediction artifacts
RESULTS_DIR = STAGE_DIR / "results" / "canonical"
ARTIFACT_PATH = RESULTS_DIR / "predictions.jsonl"

has_canonical = ARTIFACT_PATH.exists()

print(f"Canonical results available: {has_canonical}")
print(f"Artifact path: {ARTIFACT_PATH.relative_to(STAGE_DIR)}")

if has_canonical:
    print("\n⚠️ This notebook is in offline demo mode.")
    print("Canonical artifacts exist but are not loaded to avoid mixing results.")
    print("Use run_baseline.py to generate fresh predictions.")
else:
    print("\n⚠️ No canonical results found.")
    print("Run the following command to generate predictions:")
    print(f"\n  cd {STAGE_DIR}")
    print(f"  python run_baseline.py --demo {DEMO_SIZE}")

Canonical results available: False
Artifact path: results/canonical/predictions.jsonl

⚠️ No canonical results found.
Run the following command to generate predictions:

  cd /Users/veniamin/Projects/evals-chatbot/01_rag_baseline
  python run_baseline.py --demo 20


**Prediction results are not available in offline demo mode.**

After running the baseline, predictions will include:
- `case_id` - Unique case identifier
- `expected_action` - Ground truth action
- `predicted_action` - Model's routing decision
- `routing_reason` - Explanation of routing
- `retrieved_article_titles` - Top-K article titles
- `answer` - Generated answer (for answer route)
- `error` - Any execution errors

## 8. LLM-as-a-Judge Evaluation

Atomic judges evaluate answer quality on applicable dimensions.

In [14]:
# Show judge registry
print("Judge registry:")
print()
for judge_id, judge_info in JUDGES.items():
    print(f"{judge_id}:")
    print(f"  Purpose: {judge_info.get('purpose', 'N/A')}")
    print(f"  Applicability: {judge_info.get('applicability', 'N/A')}")
    print(f"  Version: {judge_info.get('version', 'N/A')}")
    print()

Judge registry:

correctness:
  Purpose: Check answer contains all required facts
  Applicability: expected_action == answer AND predicted_action == answer AND required_facts exists
  Version: correctness-v1

groundedness:
  Purpose: Check claims are supported by retrieved context
  Applicability: predicted_action == answer AND retrieved context exists
  Version: groundedness-v1

actionability:
  Purpose: Check next step clarity when required
  Applicability: predicted_action == answer
  Version: actionability-v1

conciseness:
  Purpose: Check for unnecessary repetition
  Applicability: predicted_action == answer
  Version: conciseness-v1

targeted_safety:
  Purpose: Check targeted policy violations
  Applicability: safety_type is specified (targeted cases only)
  Version: safety-v1



**Judge evaluation results are not available in offline demo mode.**

After running the baseline with judges, results will include:
- Pass/fail verdicts for each applicable judge
- Detailed reasoning for failures
- Non-applicable marking where judges don't apply
- Error tracking for judge execution failures

**Key judges:**
- `correctness` - Answer addresses the user's question
- `groundedness` - Answer is supported by retrieved context
- `actionability` - User can actually follow the guidance
- `conciseness` - Answer is appropriately brief
- `targeted_safety` - Answer handles safety issues appropriately

## 9. Evaluation Results

Overall metrics for routing, retrieval, and answer quality.

**Evaluation metrics are not available in offline demo mode.**

After running the baseline with judges, metrics will include:

**Routing metrics:**
- Routing accuracy
- Answer-route recall  
- Critical escalation recall
- Confusion matrix

**Retrieval metrics:**
- Hit@1, Hit@4
- Mean Reciprocal Rank (MRR)
- Expected-article rank

**Answer-quality metrics:**
- Correctness pass rate
- Groundedness pass rate
- Actionability pass rate
- Conciseness pass rate
- Targeted-safety pass rate

All metrics include numerator, denominator, applicable count, and excluded count.

## 10. Slice Analysis

Metrics broken down by key dimensions.

**Slice analysis is not available in offline demo mode.**

After running the baseline, slices will include:

**Dimensions:**
- By topic (cards, security, transfers, etc.)
- By difficulty (direct, ambiguous, noisy, unknown)
- By source (human seeds, synthetic variants)
- By risk level (low, critical)
- By expected action (answer, escalate)

Each slice reports:
- Case count
- Applicable count (metric-specific)
- Pass rate
- Warning when applicable < 5 cases

## 11. Failure Analysis

Representative failures by category with root cause analysis.

**Failure taxonomy:**

| Category | Description |
|----------|-------------|
| `answered_when_should_escalate` | Routed to answer on critical case |
| `escalated_when_should_answer` | Routed to escalate on low-risk case |
| `retrieval_miss` | Expected article not in top-K |
| `expected_article_outside_top_k` | Retrieved but not ranked highly |
| `incorrect_answer` | Answer doesn't address question |
| `unsupported_claim` | Answer makes claims not in KB |
| `missing_required_fact` | Answer omits essential information |
| `non_actionable_answer` | User cannot follow guidance |
| `excessive_answer` | Answer is too long/verbose |
| `safety_failure` | Inappropriate handling of safety issue |
| `judge_error` | Judge execution failed |

**Failure analysis is not available in offline demo mode.**

After running the baseline with judges, this section will show:
- Representative failures from each category
- Full query text and generated answer
- Expected vs predicted behavior
- Retrieved articles and judge reasoning
- Failure categorization and root cause

## 12. Conclusions and Next Steps

**This notebook is currently in offline demo mode.**

Conclusions require running the baseline pipeline to generate actual metrics and failures.

**To generate predictions and metrics:**

```bash
cd 01_rag_baseline
python run_baseline.py --demo 20
```

**Then reload this notebook** to see:
- Routing, retrieval, and answer-quality metrics
- Slice analysis by topic, difficulty, and variant type
- Representative failure examples with root cause
- Evidence-based conclusions and next steps

**Known limitations:**
- All labels are provisional (not human-validated)
- Judges use GPT-4o but may need calibration
- Benchmark may have blind spots in edge cases
- No measurement of false-positive escalation rate

## Appendix: Reproducibility

**Dataset and versions:**
- Benchmark: `benchmark/cases.jsonl` (355 cases)
- Knowledge base: `data/reference/revolut_help_articles.jsonl` (786 articles)
- Embeddings: `data/article_embeddings.npy` (786×1536, text-embedding-3-small)

**Prompts and judges:**
- Prompts loaded from `prompts.py` (versioned with SHA-256 hashes)
- Judges loaded from `judges.py` (atomic, single-responsibility)

**Commands:**

# Validate dataset structure
python validate_dataset.py

# Run 20-case stratified demo (after API approval)
python run_baseline.py --demo 20

# Run full canonical evaluation (355 cases)
python run_baseline.py

# Run tests
python -m pytest -q

**Current limitations:**
- Offline demo mode (no live API calls)
- Provisional labels (not human-validated)
- No multi-turn conversations
- No false-positive escalation measurement